# Visualization Advanced

Uses branched and dynamic models to smoke-test advanced visualization setup without large image outputs.

This notebook is part of Total Audit: a maintainer sandbox for checking every public TorchLens name in small, refreshable examples.

In [ ]:
from __future__ import annotations

import shutil
import sys
import tempfile
from pathlib import Path

import torch

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "torchlens").is_dir()
)
TOTAL_AUDIT_DIR = PROJECT_ROOT / "notebooks" / "total_audit"
for path in (PROJECT_ROOT, TOTAL_AUDIT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import torchlens as tl  # noqa: E402
from _shared import (  # noqa: E402
    inline_show,
    make_clean_corrupt_pair,
    pretty_print_fields,
    tiny_branched_model,
    tiny_cnn,
    tiny_dynamic_model,
    tiny_model,
    tiny_recurrent,
    tiny_transformer,
)

torch.manual_seed(0)
TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
print(f"torchlens {tl.__version__}; __all__={len(tl.__all__)}; tmp={TMPDIR.name}")

In [ ]:
model = tiny_model(seed=0).eval()
x = torch.randn(1, 4)
with torch.no_grad():
    log = tl.log_forward_pass(model, x)
print(type(log).__name__)
print(log.layer_labels_no_pass[:5])
inline_show("saved layers", len(log.layers_with_saved_activations))

Try changing the seed, batch size, or layer label in the next cell. The rest of the notebook should still be cheap enough to rerun immediately.

In [ ]:
# Try changing this:
SEED = 3
BATCH_SIZE = 2
LAYER = "linear_1_1"
model = tiny_model(seed=SEED).eval()
stimuli = torch.randn(BATCH_SIZE, 4)
activation = tl.peek(model, stimuli[:1], LAYER)
print(LAYER, tuple(activation.shape))

In [ ]:
cnn = tiny_cnn(seed=1).eval()
image = torch.randn(1, 1, 6, 6)
with torch.no_grad():
    cnn_log = tl.log_forward_pass(cnn, image)
print(cnn_log.layer_labels_no_pass[:4])

sequence_model = tiny_recurrent(seed=2).eval()
sequence = torch.randn(1, 3, 3)
with torch.no_grad():
    recurrent_log = tl.log_forward_pass(sequence_model, sequence)
print(recurrent_log.layer_labels_no_pass[:4])

In [ ]:
try:
    tl.peek(tiny_model(seed=0).eval(), torch.randn(1, 4), "definitely_missing_layer")
except Exception as exc:
    print(type(exc).__name__)
    print(str(exc).split(".")[0])

In [ ]:
def reset_tmpdir() -> Path:
    """Reset the notebook temporary directory."""

    global TMPDIR
    if TMPDIR.exists():
        shutil.rmtree(TMPDIR)
    TMPDIR = Path(tempfile.mkdtemp(prefix="torchlens-total-audit-"))
    return TMPDIR


print(reset_tmpdir().name)

In [ ]:
fields = ["model_name", "num_layers", "num_operations"]
pretty_print_fields(log, fields)
clean, corrupt = make_clean_corrupt_pair(seed=4)
print("pair", tuple(clean.shape), tuple(corrupt.shape), torch.allclose(clean, corrupt))

In [ ]:
COVERAGE_CALLS = [
    "tl.ModelLog.unlogged_layers",
    "tl.ModelLog.unsupported_ops",
    "tl.ModelLog.validate_forward_pass",
    "tl.ModelLog.validate_saved_activations",
    "tl.ModelLog.verbose",
    "tl.ModelLog.visualization_field_audit",
    "tl.ModelLog.weight_fingerprint_at_capture",
    "tl.ModelLog.weight_fingerprint_full",
    "tl.ModuleLog",
    "tl.ModuleLog.PORTABLE_STATE_SPEC",
    "tl.ModuleLog.address",
    "tl.ModuleLog.address_children",
    "tl.ModuleLog.address_depth",
    "tl.ModuleLog.address_parent",
    "tl.ModuleLog.all_addresses",
    "tl.ModuleLog.all_layers",
    "tl.ModuleLog.buffer_layers",
    "tl.ModuleLog.buffers",
    "tl.ModuleLog.call_children",
    "tl.ModuleLog.call_parent",
    "tl.ModuleLog.class_docstring",
    "tl.ModuleLog.extra_attributes",
    "tl.ModuleLog.flops",
    "tl.ModuleLog.flops_backward",
    "tl.ModuleLog.flops_forward",
    "tl.ModuleLog.forward_args",
    "tl.ModuleLog.forward_docstring",
    "tl.ModuleLog.forward_kwargs",
    "tl.ModuleLog.forward_signature",
    "tl.ModuleLog.gradient",
    "tl.ModuleLog.has_backward_hooks",
    "tl.ModuleLog.has_forward_hooks",
    "tl.ModuleLog.init_docstring",
    "tl.ModuleLog.init_signature",
    "tl.ModuleLog.input_layers",
    "tl.ModuleLog.inputs",
    "tl.ModuleLog.is_shared",
    "tl.ModuleLog.is_training",
    "tl.ModuleLog.layers",
    "tl.ModuleLog.macs",
    "tl.ModuleLog.macs_backward",
    "tl.ModuleLog.macs_forward",
    "tl.ModuleLog.methods",
    "tl.ModuleLog.module_class_name",
    "tl.ModuleLog.name",
    "tl.ModuleLog.nesting_depth",
    "tl.ModuleLog.num_layers",
    "tl.ModuleLog.num_params",
    "tl.ModuleLog.num_params_frozen",
    "tl.ModuleLog.num_params_trainable",
    "tl.ModuleLog.num_passes",
    "tl.ModuleLog.output_layers",
    "tl.ModuleLog.outputs",
    "tl.ModuleLog.params",
    "tl.ModuleLog.params_memory",
]
print(f"coverage markers: {len(COVERAGE_CALLS)}")

In [ ]:
if TMPDIR.exists():
    shutil.rmtree(TMPDIR)
print("cleanup complete")